In [8]:
import os
import cdsapi

# Local data folder for ORAS5 outputs
DATA_DIR = '../../data/ORAS5'
os.makedirs(DATA_DIR, exist_ok=True)

dataset = "reanalysis-oras5"
request = {
    "product_type": ["operational"],
    "vertical_resolution": "single_level",
    "variable": [
        "depth_of_20_c_isotherm",
        "ocean_heat_content_for_the_upper_300m",
        "sea_surface_height"
    ],
    "year": ["2024"],
    "month": [
        "01", "02", "03",
        "04", "05", "06",
        "07", "08", "09",
        "10", "11", "12"
    ],
    # Same grid and region as the ERA5 dataset (El Nino Pacific box).
    # ORAS5 does not wrap longitudes across the dateline like ERA5, so the
    # eastern bound -80 is expressed in 0-360 convention as 280.
    "grid": [2.0, 2.0],
    "area": [30, 120, -30, 280],  # N, W, S, E
    "data_format": "netcdf"
}

client = cdsapi.Client()
target = os.path.join(DATA_DIR, 'oras5_2024.zip')
client.retrieve(dataset, request, target)
print(f"Downloaded to {target}")


2026-06-25 22:03:03,611 INFO [2026-04-30T00:00:00] Please note that a dedicated catalogue entry for this dataset, post-processed and stored in Analysis Ready Cloud Optimized (ARCO) format (Zarr), is available for optimised time-series retrievals (i.e. for retrieving data from selected variables for a single point over an extended period of time in an efficient way). You can discover it [here](https://cds.climate.copernicus.eu/datasets/reanalysis-oras5-timeseries?tab=overview).
2026-06-25 22:03:03,612 INFO Request ID is 1e20138c-50bf-4b36-af06-02ac7feb363c
2026-06-25 22:03:05,691 INFO status has been updated to accepted
2026-06-25 22:03:30,815 INFO status has been updated to failed


2026-06-25 22:03:03,611 INFO [2026-04-30T00:00:00] Please note that a dedicated catalogue entry for this dataset, post-processed and stored in Analysis Ready Cloud Optimized (ARCO) format (Zarr), is available for optimised time-series retrievals (i.e. for retrieving data from selected variables for a single point over an extended period of time in an efficient way). You can discover it [here](https://cds.climate.copernicus.eu/datasets/reanalysis-oras5-timeseries?tab=overview).
2026-06-25 22:03:03,612 INFO Request ID is 1e20138c-50bf-4b36-af06-02ac7feb363c
2026-06-25 22:03:05,691 INFO status has been updated to accepted
2026-06-25 22:03:30,815 INFO status has been updated to failed


HTTPError: 400 Client Error: Bad Request for url: https://cds.climate.copernicus.eu/api/retrieve/v1/jobs/1e20138c-50bf-4b36-af06-02ac7feb363c/results
The job has failed
Area selection resulted in a dataset with zero length dimension for: y.
Please ensure that your area selection covers at least one point in the data.
The job failed with: InvalidRequest

In [ ]:
import os
import glob
import json
import zipfile
import xarray as xr

DATA_DIR = '../../data/ORAS5'
zip_path = os.path.join(DATA_DIR, 'oras5_2024.zip')

# ORAS5 downloads arrive as a zip of NetCDF files (one per variable/month)
extract_dir = os.path.join(DATA_DIR, 'nc')
os.makedirs(extract_dir, exist_ok=True)
if zipfile.is_zipfile(zip_path):
    with zipfile.ZipFile(zip_path) as z:
        z.extractall(extract_dir)
    nc_files = sorted(glob.glob(os.path.join(extract_dir, '*.nc')))
else:
    # Single NetCDF file was returned
    nc_files = [zip_path]

print(f"Found {len(nc_files)} NetCDF file(s)")

# Merge all variables/months into one dataset
ds = xr.open_mfdataset(nc_files, combine='by_coords')

# Save metadata
metadata = {
    'dimensions': dict(ds.sizes),
    'variables': {},
    'global_attributes': dict(ds.attrs)
}
for var in ds.variables:
    metadata['variables'][var] = {
        'dims': list(ds[var].dims),
        'shape': list(ds[var].shape),
        'dtype': str(ds[var].dtype),
        'attributes': dict(ds[var].attrs)
    }

meta_path = os.path.join(DATA_DIR, 'oras5_2024_metadata.json')
with open(meta_path, 'w') as f:
    json.dump(metadata, f, indent=2, default=str)
print(f"Metadata saved to {meta_path}")

# Convert to DataFrame and save as CSV
csv_path = os.path.join(DATA_DIR, 'oras5_2024.csv')
df = ds.to_dataframe().reset_index()
df.to_csv(csv_path, index=False)
print(f"CSV saved to {csv_path} ({len(df)} rows)")

ds.close()


In [ ]:
import os
import pickle
import mimetypes
from google_auth_oauthlib.flow import InstalledAppFlow
from google.auth.transport.requests import Request
from googleapiclient.discovery import build
from googleapiclient.http import MediaFileUpload

SCOPES = ['https://www.googleapis.com/auth/drive']
TOKEN_PATH = '../../token.pickle'
CLIENT_SECRETS = '../../client_secrets.json'

# Parent Drive folder under which all data folders are uploaded
DRIVE_PARENT_ID = '1zLJgkYkrM1LRgtl7v5sfAQ4aits9zfUq'

DATA_DIR = '../../data/ORAS5'

# Only upload the CSV and the metadata (two files)
FILES_TO_UPLOAD = [
    os.path.join(DATA_DIR, 'oras5_2024.csv'),
    os.path.join(DATA_DIR, 'oras5_2024_metadata.json'),
]


def get_drive_service():
    creds = None
    if os.path.exists(TOKEN_PATH):
        with open(TOKEN_PATH, 'rb') as f:
            creds = pickle.load(f)
    if not creds or not creds.valid:
        if creds and creds.expired and creds.refresh_token:
            creds.refresh(Request())
        else:
            flow = InstalledAppFlow.from_client_secrets_file(CLIENT_SECRETS, SCOPES)
            creds = flow.run_local_server(port=0)
        with open(TOKEN_PATH, 'wb') as f:
            pickle.dump(creds, f)
    return build('drive', 'v3', credentials=creds)


def get_or_create_folder(service, name, parent_id):
    """Return the Drive folder id for `name` under `parent_id`, creating it if needed."""
    query = (
        f"name = '{name}' and mimeType = 'application/vnd.google-apps.folder' "
        f"and '{parent_id}' in parents and trashed = false"
    )
    response = service.files().list(q=query, spaces='drive', fields='files(id, name)').execute()
    files = response.get('files', [])
    if files:
        return files[0]['id']
    folder = service.files().create(
        body={'name': name, 'mimeType': 'application/vnd.google-apps.folder', 'parents': [parent_id]},
        fields='id'
    ).execute()
    return folder['id']


service = get_drive_service()

# Create (or reuse) the ORAS folder to hold the CSV and metadata
oras_folder_id = get_or_create_folder(service, 'ORAS', DRIVE_PARENT_ID)
print(f"Folder: ORAS (Drive id: {oras_folder_id})")

for path in FILES_TO_UPLOAD:
    name = os.path.basename(path)
    mimetype = mimetypes.guess_type(path)[0] or 'application/octet-stream'
    media = MediaFileUpload(path, mimetype=mimetype, resumable=True)
    result = service.files().create(
        body={'name': name, 'parents': [oras_folder_id]},
        media_body=media,
        fields='id, name'
    ).execute()
    print(f"  Uploaded: {result['name']} (id: {result['id']})")

print("Done.")
